# Step 4 — Robustness to Panel Composition

Re-runs the full-sample PCA across panel subsets (dropping Japan/Canada, core-5, euro-vs-anglo) to confirm the UK conclusion is stable.

**Script notes**

```
Step 4 - Robustness of the UK conclusion to panel composition
Re-run the full-sample 10y PCA on several panels and check that:
  (a) PC1 stays a positive 'global level' factor,
  (b) UK PC1 loading stays high,
  (c) UK idiosyncratic share (1-communality_PC1) stays low / below average.
Panels:
  ALL7        : US UK DE FR IT JP CA           (baseline)
  drop_JP     : remove Japan (own BoJ regime, lowest PC1 load)
  drop_JP_CA  : also remove Canada (roll-prone investing.com)
  core5       : US UK DE FR IT  (article's own peer set: 'BIFs' + US + DE)
  euro_anglo  : UK DE FR IT     (UK vs core-euro only, the article's comparison)
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import numpy as np
import pandas as pd

In [2]:
ch = pd.read_csv("./changes_bp.csv", index_col=0, parse_dates=True)

In [3]:
def pca(cols):
    labels = [c[:2] for c in cols]
    X = ch[cols]
    Z = (X - X.mean())/X.std(ddof=1)
    C = np.corrcoef(Z.values, rowvar=False)
    ev, evec = np.linalg.eigh(C)
    o = np.argsort(ev)[::-1]; ev, evec = ev[o], evec[:, o]
    load = evec*np.sqrt(ev)
    if (load[:,0] < 0).sum() > len(labels)/2:
        load[:,0] *= -1
    comm1 = load[:,0]**2
    return labels, ev/ev.sum(), load[:,0], comm1

In [4]:
panels = {
    "ALL7":       ["US_10y","UK_10y","DE_10y","FR_10y","IT_10y","JP_10y","CA_10y"],
    "drop_JP":    ["US_10y","UK_10y","DE_10y","FR_10y","IT_10y","CA_10y"],
    "drop_JP_CA": ["US_10y","UK_10y","DE_10y","FR_10y","IT_10y"],
    "core5":      ["US_10y","UK_10y","DE_10y","FR_10y","IT_10y"],
    "euro_anglo": ["UK_10y","DE_10y","FR_10y","IT_10y"],
}

In [5]:
print("="*70)
print("STEP 4  ROBUSTNESS OF UK CONCLUSION TO PANEL COMPOSITION (10y, full sample)")
print("="*70)

STEP 4  ROBUSTNESS OF UK CONCLUSION TO PANEL COMPOSITION (10y, full sample)


In [6]:
summary = []
for name, cols in panels.items():
    labels, vshare, pc1load, comm1 = pca(cols)
    uk = labels.index("UK")
    avg_idio = (1-comm1).mean()
    uk_rank = sorted(pc1load, reverse=True).index(pc1load[uk]) + 1
    summary.append({
        "panel": name, "n_ctry": len(cols),
        "PC1_var%": round(vshare[0]*100,1),
        "all_pos_PC1": bool((pc1load>0).all()),
        "UK_PC1_load": round(pc1load[uk],3),
        "UK_rank": f"{uk_rank}/{len(cols)}",
        "UK_idio": round(1-comm1[uk],3),
        "avg_idio": round(avg_idio,3),
        "UK_vs_avg": "BELOW" if (1-comm1[uk])<avg_idio else "ABOVE",
    })

In [7]:
S = pd.DataFrame(summary).set_index("panel")
print("\n")
print(S.to_string())



            n_ctry  PC1_var%  all_pos_PC1  UK_PC1_load UK_rank  UK_idio  avg_idio UK_vs_avg
panel                                                                                      
ALL7             7      54.9         True        0.847     1/7    0.283     0.451     BELOW
drop_JP          6      60.9         True        0.869     1/6    0.246     0.391     BELOW
drop_JP_CA       5      62.0         True        0.852     1/5    0.273     0.380     BELOW
core5            5      62.0         True        0.852     1/5    0.273     0.380     BELOW
euro_anglo       4      67.7         True        0.794     4/4    0.369     0.323     ABOVE


In [8]:
print("\nInterpretation checks:")
print(f"  PC1 a positive global factor in every panel : {S['all_pos_PC1'].all()}")
print(f"  UK ranks 1st or 2nd on PC1 in every panel   : {all(int(r.split('/')[0])<=2 for r in S['UK_rank'])}")
print(f"  UK idiosyncratic BELOW average in every panel: {(S['UK_vs_avg']=='BELOW').all()}")
print(f"  UK PC1 loading range across panels          : {S['UK_PC1_load'].min():.3f} - {S['UK_PC1_load'].max():.3f}")
print(f"  UK idiosyncratic range across panels        : {S['UK_idio'].min():.3f} - {S['UK_idio'].max():.3f}")


Interpretation checks:
  PC1 a positive global factor in every panel : True
  UK ranks 1st or 2nd on PC1 in every panel   : False
  UK idiosyncratic BELOW average in every panel: False
  UK PC1 loading range across panels          : 0.794 - 0.869
  UK idiosyncratic range across panels        : 0.246 - 0.369


In [9]:
S.to_csv("./pca10_robustness.csv")
print("\nsaved pca10_robustness.csv")


saved pca10_robustness.csv
